# Launch & finish DPS runs of `run_compute_trends_ols` (v2)

Submits DPS jobs that compute per-pixel **AGB (or height) trends** over a period of years for each boreal tile: an OLS
slope (`*_ols.tif`) and Mann-Kendall trend classes (`*_kendallclasses.tif`), using the yearly AGB maps, the HLS
composites (dates) and land-cover / topography masks. It finds which tiles are still **missing** or **failed**,
resubmits only those, and after jobs finish rebuilds the two tile indexes (tindex) that define what "complete" means.

**Run top to bottom. The only cell you edit is `1. Run configuration`.** Nothing is submitted unless `DRY_RUN = False`.

| Step | What it does | Touches DPS? |
|---|---|---|
| 2. Status | Reads the TRENDCLASS and TRENDOLS tindexes: tiles **have** (in both) vs **missing** | no |
| 3. Fails | Latest job per tile from saved submission logs → which tiles **failed** (and are still not complete) | reads job status |
| 4. Plan | Tiles to submit (`DO_MISSING` / `DO_FAILS`), minus tiles that lack an input (AGB, HLS or mask for any year). Saved as CSV. | no |
| 5. Submit | Submits the plan (only if `DRY_RUN = False`) | **yes** |
| 6. Assess | Status counts for the submission just made | reads job status |
| 7. Rebuild tindex | After jobs finish: backs up, rebuilds TRENDOLS and TRENDCLASS from **all** run dates, verifies counts did not shrink | no |

**Typical cycle to finish a trend map**: `DO_MISSING=True` → plan → submit → wait → 6 → 7 → back to 2. Then `DO_FAILS=True`
for stragglers, or keep `DO_MISSING` (missing ⊇ failed once the tindex is rebuilt).

**`DO_MISSING` vs `DO_FAILS`**
* **MISSING** = in the tile set but not in **both** trend tindexes. Covers never-submitted and failed tiles, as current as
  the last tindex rebuild (step 7).
* **FAILS** = the tile's most recent job for this identifier has status `Failed` (or never submitted successfully) and the
  tile is not complete. Use right after a run, before rebuilding the tindex.
* Both: union. Neither: every tile in the tile set (a full re-run, which needs `CONFIRM_FULL_RUN = True`).

**Where this runs (MAAP):** run on the **MAAP ADE** with the `pangeo` environment (DPS submission and the tindex script
use `/opt/conda/envs/pangeo`). The algorithm code and yml live in the `icesat2_boreal` repo (`ICESAT2_BOREAL_REPO_PATH`);
keep that clone in your ADE home (`/projects/code/...`), not in a bucket mount.

Paul Montesano, PhD — v2 Oct 2026, from `icesat2_boreal/notebooks/3.Gridded_product_development/DPS_compute_trends.ipynb`,
same layout as `DPS_run_boreal_v2.ipynb`

In [ ]:
import os, sys, glob, json, shutil, datetime, subprocess
from functools import lru_cache
import pandas as pd
import geopandas as gpd

from maap.maap import MAAP
maap = MAAP(maap_host='api.maap-project.org')

# One clone of icesat2_boreal for imports (ExtractUtils, mosaiclib, compute_trends), the algorithm yml and the
# build_tindex script. The original notebook used /projects/code/icesat2_boreal.
ICESAT2_BOREAL_REPO_PATH = '/projects/code/icesat2_boreal'
sys.path.insert(0, ICESAT2_BOREAL_REPO_PATH + '/lib')
import ExtractUtils
import mosaiclib
from mosaiclib import (AGB_TINDEX_FN_DICT, HLS_TINDEX_FN_DICT, LC_TINDEX_FN_DICT, TOPO_TINDEX_FN_DICT,
                       boreal_tile_index_path)
print('ExtractUtils from', ExtractUtils.__file__)

## 1. Run configuration — the only cell to edit
Fill-in runs **must** use the same `VERSION`, period (`YEAR_LIST`), `AGB_TINDEX_KEY`, `ALGO_PARAMS` and masks as the
original run, so new tiles are consistent with the rest of the trend map.

In [ ]:
# --- What product -----------------------------------------------------------
ALGO_ID     = 'run_compute_trends_ols'
VERSION     = 'compute_trends_ols_v3'        # = algorithm_version in dps/registered/run_compute_trends_ols.yml (git tag)
USER        = 'montesano'
VAR         = 'AGB'                          # 'AGB' or 'HT': the yearly maps the trend is computed from
AGB_TINDEX_KEY = 'v3.1_multiyr'              # yearly maps: mosaiclib.AGB_TINDEX_FN_DICT[f'{year}_{AGB_TINDEX_KEY}']
YEAR_LIST   = list(range(2016, 2025 + 1))    # trend period (every year needs a map and an HLS composite)

# --- Which tiles -------------------------------------------------------------
DO_MISSING = True        # tiles in the tile set not yet in both trend tindexes
DO_FAILS   = False       # tiles whose latest job failed and are still not complete
CONFIRM_FULL_RUN = False # must be True to submit every tile in the tile set (both flags False)
TILE_GROUPS = ['north america centra', 'north america west', 'north america east']   # boreal_tiles.tile_group; None = all
TILE_SET = None          # explicit list of tile_num (overrides TILE_GROUPS), e.g. mosaiclib.set_ak_interior
IN_BOREAL_ONLY = True    # keep only tiles intersecting the WWF circumboreal polygon
REFRESH_FAILS = True     # re-query job status (slow for many jobs); False = reuse the cached fails JSON

# --- Safety -----------------------------------------------------------------
DRY_RUN  = True          # True: build and save the plan only. False: submit in step 5.
MAX_JOBS = 6000          # refuse to submit more than this in one go

# --- Algorithm parameters (copied unchanged from the original run) -----------
ALPHA_LEVEL = 'standard'  # key of compute_trends.suggest_alpha_by_temporal_resolution(n_years): conservative/standard/liberal
N_PROCESSES = 30          # 30 for the 32 GB worker, 15 for the 16 GB worker (worker chosen from this below)
ALGO_PARAMS = {
    'chunk-size': '300',
    'high-latitude-threshold': '60',   # moss/lichen masking with ESA WorldCover 2020 above this latitude
    'biomass-threshold': '10',
    'slope-threshold': '15',
}
MASK_TINDEXES = [LC_TINDEX_FN_DICT['c2020updated'], TOPO_TINDEX_FN_DICT['c2020updated_v2']]

# --- Paths --------------------------------------------------------------------
DPS_SUBMISSION_RESULTS_DIR = '/projects/my-public-bucket/dps_submission_results'
TINDEX_ROOT     = '/projects/my-public-bucket/DPS_tile_lists'                  # <TYPE>/<VERSION>/<IDENTIFIER>/
DPS_OUTPUT_ROOT = f'/projects/my-private-bucket/dps_output/{ALGO_ID}/{VERSION}'
BOREAL_POLY     = '/projects/shared-buckets/montesano/databank/arc/wwf_circumboreal_Dissolve.geojson'
PLAN_DIR        = f'{DPS_SUBMISSION_RESULTS_DIR}/plans'
for d in (DPS_SUBMISSION_RESULTS_DIR, PLAN_DIR):
    os.makedirs(d, exist_ok=True)

In [ ]:
# Derived names (do not edit)
import compute_trends                                     # only for the alpha suggestion
N_YEARS     = len(YEAR_LIST)
PERIOD      = f'{YEAR_LIST[0]}-{YEAR_LIST[-1]}'
IDENTIFIER  = f'{VAR}_{AGB_TINDEX_KEY}_{PERIOD}'           # e.g. AGB_v3.1_multiyr_2016-2025 (as in the original runs)
TINDEX_TYPES = ('TRENDCLASS', 'TRENDOLS')                 # build_tindex -t; a tile is complete when in both
ALPHA_DICT  = compute_trends.suggest_alpha_by_temporal_resolution(N_YEARS, data_type='satellite')
ALPHA       = ALPHA_DICT[ALPHA_LEVEL]
WORKER_TYPE = 'maap-dps-gedi_boreal_worker-32gb' if N_PROCESSES > 15 else 'maap-dps-gedi_boreal_worker-16gb'
MODE = '+'.join([m for m, on in (('MISSING', DO_MISSING), ('FAILS', DO_FAILS)) if on]) or 'ALL'
print(f'{ALGO_ID} {VERSION} | {IDENTIFIER} | {N_YEARS} years | alpha {ALPHA} ({ALPHA_LEVEL}; {ALPHA_DICT.get("note", "")})')
print(f'worker {WORKER_TYPE} | mode {MODE} | DRY_RUN={DRY_RUN}')

### Helper functions (no edits needed)
Parameter builder: same inputs as the original notebook (yearly AGB maps as `value-rasters`, the HLS composites of the
same years as `date-rasters`, LC + topo as `mask-rasters`), now one function per tile that fails clearly when an input is
missing for that tile.

In [ ]:
@lru_cache(maxsize=None)
def read_input_tindex(fn):
    """One input tindex (csv, local or s3), indexed by tile_num; last row wins for duplicate tiles."""
    d = pd.read_csv(fn)
    return d.drop_duplicates('tile_num', keep='last').set_index('tile_num')

def input_tindexes():
    return {'value': [AGB_TINDEX_FN_DICT[f'{y}_{AGB_TINDEX_KEY}'] for y in YEAR_LIST],
            'date':  [HLS_TINDEX_FN_DICT[str(y)] for y in YEAR_LIST],
            'mask':  list(MASK_TINDEXES)}

def tile_paths(tile_num, fns, label):
    out = []
    for fn in fns:
        t = read_input_tindex(fn)
        if tile_num not in t.index:
            raise KeyError(f'{label}: {os.path.basename(os.path.dirname(fn)) or fn}')
        out.append(t.loc[tile_num, 's3_path'])
    return out

def get_in_param_dict(tile_num):
    """All DPS parameters for one tile (matches the positional inputs of run_compute_trends_ols.yml)."""
    tx = input_tindexes()
    p = {'value-rasters': ' '.join(tile_paths(tile_num, tx['value'], f'{VAR} map')),
         'date-rasters':  ' '.join(tile_paths(tile_num, tx['date'], 'HLS composite')),
         'mask-rasters':  ' '.join(tile_paths(tile_num, tx['mask'], 'mask')),
         'output': f'boreal_{VAR.lower()}_trends_{tile_num:07}_{N_YEARS:02}yrs_{PERIOD}',
         'alpha': ALPHA, 'n-processes': str(N_PROCESSES)}
    p.update(ALGO_PARAMS)
    return p

In [ ]:
def tindex_paths(ttype):
    d = f'{TINDEX_ROOT}/{ttype}/{VERSION}/{IDENTIFIER}'
    return {'dir': d, 'csv': f'{d}/{ttype}_tindex_master.csv', 'gpkg': f'{d}/{ttype}_tindex_master.gpkg'}

def read_tindex_tiles(ttype):
    """Tiles present in one trend tindex (empty set if no tindex yet)."""
    tp = tindex_paths(ttype)
    for fn in (tp['csv'], tp['gpkg']):
        if os.path.exists(fn):
            df = pd.read_csv(fn) if fn.endswith('.csv') else gpd.read_file(fn)
            return set(df.tile_num.astype(int))
    return set()

def complete_tiles():
    """A tile is complete when it is in every trend tindex (TRENDCLASS and TRENDOLS)."""
    sets = [read_tindex_tiles(t) for t in TINDEX_TYPES]
    return set.intersection(*sets) if sets else set()

def submission_logs():
    """All saved submission CSVs for this identifier, oldest first (by the timestamp in the file name).
    Matches v2 logs and the original notebook's logs (same naming: DPS_<alg>_<run>_submission_results_<n>_<time>.csv)."""
    fns = glob.glob(f'{DPS_SUBMISSION_RESULTS_DIR}/DPS_{ALGO_ID}_{IDENTIFIER}_submission_results_*.csv')
    fns = [f for f in fns if 'job_status' not in f]
    return sorted(fns, key=lambda f: int(os.path.splitext(f)[0].split('_')[-1]))

def latest_jobs():
    """One row per tile: the most recent submission for this identifier."""
    parts = []
    for order, fn in enumerate(submission_logs()):
        d = pd.read_csv(fn)
        d['log_file'], d['log_order'] = os.path.basename(fn), order
        parts.append(d)
    if not parts:
        return pd.DataFrame(columns=['tile_num', 'job_id', 'submit_status', 'log_file'])
    d = pd.concat(parts, ignore_index=True)
    d['tile_num'] = d.tile_num.astype(int)
    return d.sort_values(['log_order', 'dps_num']).drop_duplicates('tile_num', keep='last').reset_index(drop=True)

def job_status_table(jobs):
    """Adds a 'status' column via MAAP. Tiles that never got a job id are marked 'SubmitFailed'."""
    ok = jobs.job_id.notna() & (jobs.job_id.astype(str) != '')
    out = jobs.copy()
    out['status'] = 'SubmitFailed'
    if ok.any():
        st = ExtractUtils.BUILD_TABLE_JOBSTATUS(jobs[ok][['tile_num', 'job_id']])
        out.loc[ok, 'status'] = out.loc[ok, 'job_id'].map(dict(zip(st.job_id, st.status))).fillna('Unknown')
    return out

In [ ]:
def build_plan(tile_set, have, failed):
    """Tiles to submit, with the reason each tile is (or is not) included."""
    missing = sorted(set(tile_set) - have)
    want = set()
    if DO_MISSING:
        want |= set(missing)
    if DO_FAILS:
        want |= set(failed)
    if not (DO_MISSING or DO_FAILS):
        want = set(tile_set)
    rows = []
    for t in sorted(want):
        try:
            get_in_param_dict(t)
            runnable, why = True, ''
        except KeyError as e:
            runnable, why = False, f'no input for tile: {e.args[0]}'
        rows.append({'identifier': IDENTIFIER, 'tile_num': t, 'is_missing': t in missing, 'is_failed': t in failed,
                     'complete': t in have, 'runnable': runnable, 'reason_not_runnable': why})
    return pd.DataFrame(rows, columns=['identifier', 'tile_num', 'is_missing', 'is_failed', 'complete', 'runnable', 'reason_not_runnable'])

def submit_tiles(tiles):
    rows = []
    for i, t in enumerate(tiles, 1):
        try:
            r = maap.submitJob(identifier=IDENTIFIER, algo_id=ALGO_ID, version=VERSION, username=USER, queue=WORKER_TYPE,
                               **get_in_param_dict(t))
            job_id, status = r.id, r.status
        except Exception as e:
            job_id, status = None, f'error: {e}'
        rows.append({'dps_num': i, 'tile_num': t, 'submit_time': datetime.datetime.now().strftime('%Y-%m-%d-%H-%M-%S'),
                     'algo_id': ALGO_ID, 'user': USER, 'worker_type': WORKER_TYPE, 'job_id': job_id,
                     'submit_status': status, 'IDENTIFIER': IDENTIFIER, 'run_name': IDENTIFIER})
        if i in (1, len(tiles)) or i % 100 == 0:
            print(f'  {i}/{len(tiles)} tile {t}: {status}')
    df = pd.DataFrame(rows)
    now = pd.Timestamp.now().strftime('%Y%m%d%H%M')
    fn = f'{DPS_SUBMISSION_RESULTS_DIR}/DPS_{ALGO_ID}_{IDENTIFIER}_submission_results_{len(df)}_{now}.csv'
    df.to_csv(fn, index=False)
    print(f'  saved {fn}')
    return df

In [ ]:
def dps_run_dates():
    """Years, months and day range of every DPS run-date folder: dps_output/<alg>/<version>/<identifier>/YYYY/MM/DD."""
    root = f'{DPS_OUTPUT_ROOT}/{IDENTIFIER}'
    years, months, days = set(), set(), set()
    if os.path.isdir(root):
        for y in os.listdir(root):
            if not (y.isdigit() and os.path.isdir(f'{root}/{y}')):
                continue
            for m in os.listdir(f'{root}/{y}'):
                if not (m.isdigit() and os.path.isdir(f'{root}/{y}/{m}')):
                    continue
                for d in os.listdir(f'{root}/{y}/{m}'):
                    if d.isdigit():
                        years.add(y); months.add(m); days.add(int(d))
    return sorted(years), sorted(months), (min(days), max(days)) if days else (1, 31)

def rebuild_tindex(ttype, ENV='pangeo', DEBUG=False,
                   SCRIPT=f'{ICESAT2_BOREAL_REPO_PATH}/lib/build_tindex_master_refactored.py'):
    """Back up, rebuild one trend tindex (TRENDOLS: *_ols.tif, TRENDCLASS: *_kendallclasses.tif) from ALL run dates,
    verify the tile count.

    The script OVERWRITES {TYPE}_tindex_master.csv/.gpkg with only the outputs found under the run dates it is given
    (<user>/dps_output/<alg>/<VERSION>/<IDENTIFIER>/{year}/{month}/{day}/**), keeping the newest file per tile, so every
    run date is passed. (The original notebook passed fixed run years '2025 2026' and all months.)"""
    tp = tindex_paths(ttype)
    years, months, (d_min, d_max) = dps_run_dates()
    if not years:
        print(f'  no DPS output folders under {DPS_OUTPUT_ROOT}/{IDENTIFIER}; skipping')
        return None
    n_before = len(read_tindex_tiles(ttype))
    stamp = pd.Timestamp.now().strftime('%Y%m%d%H%M%S')
    bdir = f"{tp['dir']}/backup_{stamp}"
    for fn in (tp['csv'], tp['gpkg']):
        if os.path.exists(fn):
            os.makedirs(bdir, exist_ok=True)
            shutil.copy2(fn, bdir)
    os.makedirs(tp['dir'], exist_ok=True)
    cmd = [f'/opt/conda/envs/{ENV}/bin/python', SCRIPT,
           '-t', ttype, '-alg_name', ALGO_ID, '--user', USER,
           '--dps_identifier', f'{VERSION}/{IDENTIFIER}',
           '-y_list', *years, '-m_list', *months, '-d_min', str(d_min), '-d_max', str(d_max),
           '--outdir', tp['dir'], '-boreal_tile_index_path', boreal_tile_index_path,
           '--RETURN_DUPS', '--WRITE_TINDEX_MATCHES_GDF']
    if DEBUG:
        cmd.append('--DEBUG')
    print(f'  {ttype}: run dates years {years} months {months} days {d_min}-{d_max}')
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:   # the script exits 1 before writing when nothing is found, so the tindex is untouched
        print(res.stdout[-3000:], res.stderr[-3000:])
        raise RuntimeError(f'{os.path.basename(SCRIPT)} failed for {ttype} {IDENTIFIER}; backup in {bdir}')
    if DEBUG:
        print(res.stdout[-3000:])
    n_after = len(read_tindex_tiles(ttype))
    flag = 'OK' if n_after >= n_before else '!! SHRANK — restore with restore_tindex(ttype, backup_dir)'
    print(f'  {ttype}: tiles before {n_before} -> after {n_after}  {flag}   (backup: {bdir if n_before else "none"})')
    return {'tindex': ttype, 'identifier': IDENTIFIER, 'n_before': n_before, 'n_after': n_after, 'backup_dir': bdir}

def restore_tindex(ttype, backup_dir):
    for fn in glob.glob(f'{backup_dir}/*'):
        shutil.copy2(fn, tindex_paths(ttype)['dir'])
    print('restored', ttype, IDENTIFIER, 'from', backup_dir)

## 2. Status: which tiles are complete and which are missing
A tile is **complete** when it is in both trend tindexes. Tiles in only one of them (a partial output) count as missing.

In [ ]:
boreal_tiles = gpd.read_file(boreal_tile_index_path)
if TILE_SET is not None:
    cand = boreal_tiles[boreal_tiles.tile_num.isin(TILE_SET)]
elif TILE_GROUPS is not None:
    cand = boreal_tiles[boreal_tiles.tile_group.isin(TILE_GROUPS)]
    unknown = set(TILE_GROUPS) - set(boreal_tiles.tile_group)
    if unknown:
        print('Not a tile_group (check spelling):', sorted(unknown), '| available:', sorted(boreal_tiles.tile_group.dropna().unique()))
else:
    cand = boreal_tiles
if IN_BOREAL_ONLY:
    bp = gpd.read_file(BOREAL_POLY).to_crs(boreal_tiles.crs)
    boreal_poly = bp.union_all() if hasattr(bp, 'union_all') else bp.unary_union   # geopandas < 1.0
    cand = cand[cand.intersects(boreal_poly)]
TILES = sorted(int(t) for t in cand.tile_num)

TINDEX_TILES = {t: read_tindex_tiles(t) for t in TINDEX_TYPES}
HAVE = complete_tiles()
status = pd.DataFrame([{'identifier': IDENTIFIER, 'n_target': len(TILES),
                        **{f'n_in_{t}': len(set(TILES) & s) for t, s in TINDEX_TILES.items()},
                        'n_complete': len(set(TILES) & HAVE), 'n_missing': len(set(TILES) - HAVE)}])
status

In [ ]:
# Map: tiles in the target set (gray) and missing (red), on the ESRI Light Gray basemap
m = boreal_tiles[boreal_tiles.tile_num.isin(TILES)].explore(color='gray', style_kwds={'fillOpacity': 0.1, 'weight': 0.5},
                                                           tiles='Esri.WorldGrayCanvas', name='target tiles')
missing_gdf = boreal_tiles[boreal_tiles.tile_num.isin(set(TILES) - HAVE)]
missing_gdf.explore(m=m, color='red', tooltip=['tile_num', 'tile_group'], name='missing') if len(missing_gdf) else m

## 3. Fails: latest job per tile from the saved submission logs
Only tiles whose **most recent** job for the identifier failed (or never got a job id), and that are still not complete.
Cached to JSON so `DO_FAILS` works when you come back without re-querying (`REFRESH_FAILS = False`).

In [ ]:
FAILS_JSON = f'{PLAN_DIR}/fails_{ALGO_ID}_{VERSION}_{IDENTIFIER}.json'
if DO_FAILS and not REFRESH_FAILS and os.path.exists(FAILS_JSON):
    FAILED = set(json.load(open(FAILS_JSON)))
    print('loaded cached fails from', FAILS_JSON)
elif DO_FAILS:
    jobs = latest_jobs()
    if jobs.empty:
        FAILED = set(); print(f'{IDENTIFIER}: no submission logs')
    else:
        job_table = job_status_table(jobs)
        bad = job_table.status.isin(['Failed', 'SubmitFailed', 'Offline', 'Deleted'])
        FAILED = set(job_table.tile_num[bad].astype(int)) - HAVE
        print(f'{IDENTIFIER}: latest jobs {len(job_table)} | ' + ', '.join(f'{k} {v}' for k, v in job_table.status.value_counts().items())
              + f' | failed & not complete: {len(FAILED)}')
    json.dump(sorted(FAILED), open(FAILS_JSON, 'w'))
else:
    FAILED = set()
    print('DO_FAILS = False: fails not computed')

## 4. Plan: what will be submitted
Tiles without an input for **any** year of the period (no AGB map or HLS composite that year, or no mask tile) are listed as
not runnable and are **not** submitted, so they don't fail repeatedly. Review this table before setting `DRY_RUN = False`.

In [ ]:
if not (DO_MISSING or DO_FAILS) and not CONFIRM_FULL_RUN:
    raise ValueError('Both DO_MISSING and DO_FAILS are False (full re-run of the tile set). Set CONFIRM_FULL_RUN = True if intended.')

plan = build_plan(TILES, HAVE, FAILED)
plan = plan.astype({'tile_num': int, 'is_missing': bool, 'is_failed': bool, 'complete': bool, 'runnable': bool})
plan_summary = pd.DataFrame([{'identifier': IDENTIFIER, 'n_selected': len(plan), 'n_missing': int(plan.is_missing.sum()),
                              'n_failed': int(plan.is_failed.sum()), 'n_not_runnable': int((~plan.runnable).sum()),
                              'n_to_submit': int(plan.runnable.sum())}])
PLAN_FN = f'{PLAN_DIR}/plan_{ALGO_ID}_{IDENTIFIER}_{MODE}_{pd.Timestamp.now():%Y%m%d%H%M}.csv'
plan.to_csv(PLAN_FN, index=False)
print('plan saved:', PLAN_FN, '| total to submit:', int(plan.runnable.sum()))
plan_summary

In [ ]:
# Not-runnable tiles and why (fix inputs first, e.g. redo an HLS composite or AGB map, or accept as permanent gaps)
plan[~plan.runnable].groupby('reason_not_runnable').tile_num.apply(lambda s: sorted(s)).reset_index()

## 5. Submit (only if `DRY_RUN = False`)

In [ ]:
SUBMITTED = None
n_total = int(plan.runnable.sum())
if DRY_RUN:
    print(f'DRY_RUN: would submit {n_total} jobs. Set DRY_RUN = False in section 1 and re-run from the top.')
elif n_total > MAX_JOBS:
    raise ValueError(f'{n_total} jobs > MAX_JOBS={MAX_JOBS}. Raise MAX_JOBS if intended.')
else:
    tiles = plan[plan.runnable].tile_num.tolist()
    print(f'{IDENTIFIER}: {len(tiles)} jobs')
    if tiles:
        SUBMITTED = submit_tiles(tiles)

## 6. Assess the submission just made
Re-run this cell until nothing is `Accepted`/`Running`. To assess an older run in a new session, set `DO_FAILS = True` and
run section 3, which reads the saved logs.

In [ ]:
if SUBMITTED is not None:
    assess = job_status_table(SUBMITTED)
    display(assess.groupby(['IDENTIFIER', 'status']).size().unstack(fill_value=0))
else:
    print('Nothing submitted in this session.')

## 7. Rebuild the tindexes (after jobs finish)
Rebuilds **TRENDOLS** and **TRENDCLASS** from all run-date folders, backs up each existing tindex first, and reports
before/after tile counts. Then go back to section 2. The mosaic JSONs used for maps (`mosaiclib.MISC_MOSAIC_JSON_FN_DICT`
keys `TRENDOLS_<IDENTIFIER>`, `TRENDCLASS_<IDENTIFIER>`) are built from these tindexes.

In [ ]:
DO_REBUILD_TINDEX = False   # set True after jobs finish
if DO_REBUILD_TINDEX:
    rebuilt = [rebuild_tindex(t) for t in TINDEX_TYPES]
    display(pd.DataFrame([r for r in rebuilt if r]))

---
## Appendix: reference calls (not part of the run)

**Before a new algorithm version:** tag the `icesat2_boreal` commit that runs it and use the tag as `algorithm_version` in
`dps/registered/run_compute_trends_ols.yml` (and as `VERSION` above): `git add`, `git commit -m '…'`,
`git tag -f compute_trends_ols_v3`, `git push`, `git push origin -f compute_trends_ols_v3`; check with `git log` that the
tag is where origin is. Then register (below). Outputs land in `dps_output/<ALGO_ID>/<VERSION>/<IDENTIFIER>/`.

In [ ]:
SHOW_REFERENCE = False
if SHOW_REFERENCE:
    print([a for a in maap.listAlgorithms().json()['algorithms'] if ALGO_ID in a['type']])
    print(maap.getQueues().json()['queues'])
    # Register the algorithm (only after changing the yml / code):
    # maap.register_algorithm_from_yaml_file(f'{ICESAT2_BOREAL_REPO_PATH}/dps/registered/run_compute_trends_ols.yml').text
    # Alpha suggestions for this period length:
    print(ALPHA_DICT)
    # Inspect the full parameter dict for one tile:
    print(json.dumps(get_in_param_dict(TILES[0]), indent=1, default=str))